# 03 Image Scraping

## Goal
This notebook builds an image dataset from the literary text processed in the previous notebook.

The notebook will:
- load the processed literary units
- construct image search queries from city groups and keywords
- create image folders for each text unit
- download reference images
- save image metadata for later analysis and generation

In [27]:
#!pip install duckduckgo-search requests pillow

## Step 1 — Import libraries
Import the libraries needed for file handling, metadata management, image downloading, and search.

In [28]:
#!pip install ddgs

In [29]:
from pathlib import Path
import json
import re
import time
import hashlib
import requests
import pandas as pd

from ddgs import DDGS
from PIL import Image
from io import BytesIO

## Step 2 — Load the project config
Load the shared project paths created in Notebook 01.

In [30]:
PROJECT_ROOT = Path.cwd().resolve().parent
CONFIG_PATH = PROJECT_ROOT / "src" / "project_config.json"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

RAW_IMAGES_DIR = Path(config["raw_images_dir"])
PROCESSED_LITERARY_DIR = Path(config["processed_literary_dir"])
OUTPUT_TABLES_DIR = Path(config["output_tables_dir"])

print("Raw images dir:", RAW_IMAGES_DIR)
print("Processed literary dir:", PROCESSED_LITERARY_DIR)

Raw images dir: D:\Work\Workspace\Projects\Python\data-driven-surface\data\raw\scraped_images
Processed literary dir: D:\Work\Workspace\Projects\Python\data-driven-surface\data\processed\literary_text


## Step 3 — Load the processed literary units
Read the CSV file created in Notebook 02.

In [31]:
processed_csv_path = PROCESSED_LITERARY_DIR / "invisible_cities_units_processed.csv"

if not processed_csv_path.exists():
    raise FileNotFoundError(
        f"Processed literary CSV not found: {processed_csv_path}\n"
        "Please run 02_literary_text_processing.ipynb first."
    )

unit_df = pd.read_csv(processed_csv_path)
unit_df.head()

,unit_id,city_group,text,clean_text,char_count,word_count,keywords,dominant_emotion,joy,sadness,fear,awe,desire
0,1,Cities & Memory 1,Leaving there and proceeding for three days to...,Leaving there and proceeding for three days to...,745,136,"['evening', 'leaving', 'proceeding', 'east', '...",joy,2,0,0,1,1
1,2,Cities & Memory 2,When a man rides a long time through wild regi...,When a man rides a long time through wild regi...,742,131,"['isidora', 'spiral', 'young', 'rides', 'wild'...",desire,0,0,0,0,1
2,3,Cities & Desire 1,There are two ways of describing the city of D...,There are two ways of describing the city of D...,1288,221,"['dorothea', 'quarters', 'morning', 'desert', ...",desire,0,0,0,0,1
3,4,Cities & Memory 3,"In vain, great-hearted Kublai, shall I attempt...","In vain, great-hearted Kublai, shall I attempt...",1684,306,"['past', 'zaira', 'tell', 'steps', 'streets', ...",neutral,0,0,0,0,0
4,5,Cities & Desire 2,"At the end of three days, moving southward, yo...","At the end of three days, moving southward, yo...",1368,248,"['anastasia', 'desire', 'sometimes', 'agate', ...",desire,1,0,0,0,4


## Step 4 — Convert keyword strings back into Python lists
When CSV files are reloaded, keyword lists may appear as strings.
This step converts them back into list format if needed.

In [32]:
def parse_keyword_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    value = str(value).strip()

    if value.startswith("[") and value.endswith("]"):
        try:
            import ast
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass

    return []

unit_df["keywords"] = unit_df["keywords"].apply(parse_keyword_list)
unit_df[["unit_id", "city_group", "keywords"]].head()

,unit_id,city_group,keywords
0,1,Cities & Memory 1,"[evening, leaving, proceeding, east, reach, di..."
1,2,Cities & Memory 2,"[isidora, spiral, young, rides, wild, regions,..."
2,3,Cities & Desire 1,"[dorothea, quarters, morning, desert, caravan,..."
3,4,Cities & Memory 3,"[past, zaira, tell, steps, streets, telling, h..."
4,5,Cities & Desire 2,"[anastasia, desire, sometimes, agate, onyx, ch..."


## Step 5 — Define query construction rules
Build compact and image-friendly search queries from:
- city group
- top keywords
- optional emotional tone

In [33]:
visual_priority_words = {
    "towers", "walls", "streets", "desert", "caravan", "domes", "bronze",
    "statues", "spiral", "staircases", "quarters", "lamppost", "dock",
    "agate", "onyx", "concentric", "trees", "stones", "blind", "puma",
    "ship", "canals", "market", "shells", "telescopes", "buildings",
    "routes", "houses", "railing", "gods", "silver"
}

def select_visual_keywords(keywords, max_keywords=5):
    if not isinstance(keywords, list):
        return []

    strong = [k for k in keywords if k in visual_priority_words]
    remaining = [k for k in keywords if k not in strong]

    selected = strong[:max_keywords]
    if len(selected) < max_keywords:
        selected.extend(remaining[: max_keywords - len(selected)])

    return selected[:max_keywords]

def build_search_query(row):
    city_group = str(row["city_group"])
    keywords = select_visual_keywords(row["keywords"], max_keywords=4)
    emotion = str(row["dominant_emotion"]).lower()

    keyword_text = ", ".join(keywords)

    query = f"surreal city architecture, {keyword_text}, {emotion} atmosphere"
    return query

## Step 6 — Build search queries for all units
Create one image-search query per literary unit.

In [34]:
unit_df["search_query"] = unit_df.apply(build_search_query, axis=1)
unit_df[["unit_id", "city_group", "search_query"]].head(10)

,unit_id,city_group,search_query
0,1,Cities & Memory 1,"surreal city architecture, silver, domes, bron..."
1,2,Cities & Memory 2,"surreal city architecture, spiral, buildings, ..."
2,3,Cities & Desire 1,"surreal city architecture, quarters, desert, c..."
3,4,Cities & Memory 3,"surreal city architecture, streets, lamppost, ..."
4,5,Cities & Desire 2,"surreal city architecture, agate, onyx, concen..."
5,6,Cities & Signs 1,"surreal city architecture, trees, stones, stre..."
6,7,Cities & Memory 4,"surreal city architecture, streets, houses, zo..."
7,8,Cities & Desire 3,"surreal city architecture, desert, ship, camel..."
8,9,Cities & Signs 2,"surreal city architecture, blind, puma, zirma,..."
9,10,Thin Cities 1,"surreal city architecture, marco, past, khan, ..."


## Step 7 — Inspect example queries
Preview a few generated queries before scraping.

In [35]:
for _, row in unit_df[["unit_id", "city_group", "search_query"]].head(5).iterrows():
    print(f"{row['unit_id']} | {row['city_group']}")
    print(row["search_query"])
    print("-" * 80)

1 | Cities & Memory 1
surreal city architecture, silver, domes, bronze, statues, joy atmosphere
--------------------------------------------------------------------------------
2 | Cities & Memory 2
surreal city architecture, spiral, buildings, staircases, isidora, desire atmosphere
--------------------------------------------------------------------------------
3 | Cities & Desire 1
surreal city architecture, quarters, desert, caravan, routes, desire atmosphere
--------------------------------------------------------------------------------
4 | Cities & Memory 3
surreal city architecture, streets, lamppost, railing, dock, neutral atmosphere
--------------------------------------------------------------------------------
5 | Cities & Desire 2
surreal city architecture, agate, onyx, concentric, anastasia, desire atmosphere
--------------------------------------------------------------------------------


## Step 8 — Create a folder structure for downloaded images
Each literary unit will receive its own image folder.

In [36]:
def safe_folder_name(text: str) -> str:
    text = re.sub(r"[^\w\s\-&]", "", text)
    text = re.sub(r"\s+", "_", text.strip())
    return text

for row in unit_df.itertuples(index=False):
    folder_name = f"{row.unit_id:02d}_{safe_folder_name(row.city_group)}"
    folder_path = RAW_IMAGES_DIR / folder_name
    folder_path.mkdir(parents=True, exist_ok=True)

print("Image folders created.")

Image folders created.


## Step 9 — Define image search and download helpers
This step:
- searches for image results
- downloads the images
- checks that they are valid image files
- stores them in the appropriate folder

In [37]:
def search_images(query, max_results=6, retries=3, sleep_seconds=3):
    results = []

    for attempt in range(retries):
        try:
            with DDGS() as ddgs:
                search_results = ddgs.images(
                    query,
                    max_results=max_results,
                    safesearch="off"
                )
                results = list(search_results)

            if results:
                return results

        except Exception as e:
            print(f"Search failed for query: {query}")
            print(f"Attempt {attempt + 1}/{retries}")
            print("Error:", e)

            if attempt < retries - 1:
                time.sleep(sleep_seconds)

    return results

## Step 10 — Download images for a small test subset
Start with a small subset before scraping the full dataset.

In [ ]:
test_subset = unit_df.head(3).copy()
metadata_records = []

for row in test_subset.itertuples(index=False):
    folder_name = f"{row.unit_id:02d}_{safe_folder_name(row.city_group)}"
    folder_path = RAW_IMAGES_DIR / folder_name

    results = search_images(row.search_query, max_results=6)

    print(f"\nUnit {row.unit_id} | {row.city_group}")
    print("Query:", row.search_query)
    print("Results found:", len(results))

    image_count = 0

    for idx, result in enumerate(results, start=1):
        image_url = result.get("image")
        source_url = result.get("url", "").lower()
        title = result.get("title", "").lower()

        if not image_url:
            continue

        if any(x in source_url for x in ["artstation", "pinterest", "deviantart", "pixiv"]):
            continue

        if any(x in title for x in ["illustration", "drawing", "painting", "concept art", "sketch"]):
            continue

        filename = f"img_{idx:02d}.jpg"
        save_path = folder_path / filename

        success, error = download_image(image_url, save_path)

        metadata_records.append({
            "unit_id": row.unit_id,
            "city_group": row.city_group,
            "search_query": row.search_query,
            "image_rank": idx,
            "image_url": image_url,
            "source_url": source_url,
            "title": title,
            "local_path": str(save_path),
            "download_success": success,
            "error": error
        })

        if success:
            image_count += 1

    print("Images downloaded:", image_count)
    time.sleep(5)


Unit 1 | Cities & Memory 1
Query: surreal city architecture, silver, domes, bronze, statues, joy atmosphere
Results found: 6
Images downloaded: 6

Unit 2 | Cities & Memory 2
Query: surreal city architecture, spiral, buildings, staircases, isidora, desire atmosphere
Results found: 6
Images downloaded: 5

Unit 3 | Cities & Desire 1
Query: surreal city architecture, quarters, desert, caravan, routes, desire atmosphere
Results found: 6
Images downloaded: 6


## Step 11 — Review test download metadata
Inspect the metadata for the first test scraping run.

In [39]:
metadata_df = pd.DataFrame(metadata_records)
metadata_df.head(10)

,unit_id,city_group,search_query,image_rank,image_url,source_url,title,local_path,download_success,error
0,1,Cities & Memory 1,"surreal city architecture, silver, domes, bron...",1,https://pics.craiyon.com/2023-10-24/332aa5adef...,https://www.craiyon.com/image/BtAnG3xCRzWOAD7q...,Cityscape with silver domes and bronze statues...,D:\Work\Workspace\Projects\Python\data-driven-...,True,None
1,1,Cities & Memory 1,"surreal city architecture, silver, domes, bron...",2,https://pics.craiyon.com/2024-04-28/oTRPsNoNT5...,https://www.craiyon.com/image/Vmzk7YtkTdWm6uDu...,"City with silver domes, bronze statues, paved ...",D:\Work\Workspace\Projects\Python\data-driven-...,True,None
2,1,Cities & Memory 1,"surreal city architecture, silver, domes, bron...",3,https://pics.craiyon.com/2023-10-07/d79e0b758b...,https://www.craiyon.com/search/aerial+view+of+...,aerial view of a city with silver domes and br...,D:\Work\Workspace\Projects\Python\data-driven-...,True,None
3,1,Cities & Memory 1,"surreal city architecture, silver, domes, bron...",4,https://pics.craiyon.com/2023-10-30/8aaacd0020...,https://www.craiyon.com/image/0n7cZEu6QHKsOuk0...,Mystical city with shining domes and statues,D:\Work\Workspace\Projects\Python\data-driven-...,True,None
4,1,Cities & Memory 1,"surreal city architecture, silver, domes, bron...",5,https://pics.craiyon.com/2024-12-15/Sykd0BPzRr...,https://www.craiyon.com/image/_Y37EuUXQaaJyzd2...,City with silver domes and bronze statues on a...,D:\Work\Workspace\Projects\Python\data-driven-...,True,None
5,1,Cities & Memory 1,"surreal city architecture, silver, domes, bron...",6,https://pics.craiyon.com/2023-10-30/e26901fe03...,https://www.craiyon.com/image/sHf6p4CaQs-IopNK...,Fantasy city with silver domes and bronze statues,D:\Work\Workspace\Projects\Python\data-driven-...,True,None
6,2,Cities & Memory 2,"surreal city architecture, spiral, buildings, ...",1,https://images.stockcake.com/public/8/e/d/8ed0...,https://stockcake.com/i/spiral-urban-dreamscap...,"Free Spiral Urban Dreamscape Image - Spiral, S...",D:\Work\Workspace\Projects\Python\data-driven-...,True,None
7,2,Cities & Memory 2,"surreal city architecture, spiral, buildings, ...",2,https://i.pinimg.com/originals/9b/28/18/9b2818...,https://www.pinterest.com/pin/isidora-nine-inv...,Isidora Nine Invisible cities. Chapter 1. Citi...,D:\Work\Workspace\Projects\Python\data-driven-...,True,None
8,2,Cities & Memory 2,"surreal city architecture, spiral, buildings, ...",3,https://cdn.openart.ai/published/uNXcBp8dNKILa...,https://openart.ai/search/architecture,OpenArt - Find and Easily Create Customized ar...,D:\Work\Workspace\Projects\Python\data-driven-...,True,None
9,2,Cities & Memory 2,"surreal city architecture, spiral, buildings, ...",4,https://images.stockcake.com/public/1/0/3/103c...,https://stockcake.com/i/surreal-spiral-stairca...,Free Surreal spiral staircase Image | Download...,D:\Work\Workspace\Projects\Python\data-driven-...,True,None


## Step 12 — Expand scraping to the full dataset
If the test subset looks correct, apply the same process to all units.

In [40]:
download_all = True  # Change to True when ready

In [41]:
if download_all:
    metadata_records = []

    for row in unit_df.itertuples(index=False):
        folder_name = f"{row.unit_id:02d}_{safe_folder_name(row.city_group)}"
        folder_path = RAW_IMAGES_DIR / folder_name

        results = search_images(row.search_query, max_results=3)

        print(f"\nUnit {row.unit_id} | {row.city_group}")
        print("Query:", row.search_query)
        print("Results found:", len(results))

        image_count = 0

        for idx, result in enumerate(results, start=1):
            image_url = result.get("image")
            source_url = result.get("url", "")
            title = result.get("title", "")

            if not image_url:
                continue

            filename = f"img_{idx:02d}.jpg"
            save_path = folder_path / filename

            success, error = download_image(image_url, save_path)

            metadata_records.append({
                "unit_id": row.unit_id,
                "city_group": row.city_group,
                "search_query": row.search_query,
                "image_rank": idx,
                "image_url": image_url,
                "source_url": source_url,
                "title": title,
                "local_path": str(save_path),
                "download_success": success,
                "error": error
            })

            if success:
                image_count += 1

        print("Images downloaded:", image_count)
        time.sleep(2)

    metadata_df = pd.DataFrame(metadata_records)
else:
    print("Full scraping is disabled. Set download_all = True to run it.")


Unit 1 | Cities & Memory 1
Query: surreal city architecture, silver, domes, bronze, statues, joy atmosphere
Results found: 3
Images downloaded: 3

Unit 2 | Cities & Memory 2
Query: surreal city architecture, spiral, buildings, staircases, isidora, desire atmosphere
Results found: 3
Images downloaded: 3

Unit 3 | Cities & Desire 1
Query: surreal city architecture, quarters, desert, caravan, routes, desire atmosphere
Results found: 3
Images downloaded: 3

Unit 4 | Cities & Memory 3
Query: surreal city architecture, streets, lamppost, railing, dock, neutral atmosphere
Results found: 3
Images downloaded: 3

Unit 5 | Cities & Desire 2
Query: surreal city architecture, agate, onyx, concentric, anastasia, desire atmosphere
Results found: 3
Images downloaded: 3

Unit 6 | Cities & Signs 1
Query: surreal city architecture, trees, stones, streets, statues, sadness atmosphere
Results found: 3
Images downloaded: 3

Unit 7 | Cities & Memory 4
Query: surreal city architecture, streets, houses, zora,

## Step 13 — Save scraping metadata
Export the metadata table for later filtering and inspection.

In [42]:
metadata_csv_path = OUTPUT_TABLES_DIR / "image_scraping_metadata.csv"
metadata_df.to_csv(metadata_csv_path, index=False, encoding="utf-8-sig")

print("Saved metadata to:", metadata_csv_path)

Saved metadata to: D:\Work\Workspace\Projects\Python\data-driven-surface\outputs\tables\image_scraping_metadata.csv


## Step 14 — Summarise the scraping results
Check how many images were successfully downloaded.

In [43]:
if not metadata_df.empty:
    print("Total records:", len(metadata_df))
    print("Successful downloads:", metadata_df["download_success"].sum())
    print("Failed downloads:", (~metadata_df["download_success"]).sum())
else:
    print("No metadata available yet.")

Total records: 135
Successful downloads: 132
Failed downloads: 3


## Step 15 — Optional: inspect successful images only
Filter the metadata table to show only successfully downloaded files.

In [44]:
if not metadata_df.empty:
    success_df = metadata_df[metadata_df["download_success"] == True].copy()
    success_df.head(20)
else:
    print("No metadata available yet.")

## Step 16 — Summary
This notebook has:
- created image search queries from literary units
- created image folders for each city-text unit
- downloaded a test image dataset
- saved image metadata for later use

Next notebook:
- `04_ai_text_generation.ipynb`